In [9]:
import os
import pandas as pd
import sqlite3

os.chdir(os.path.expanduser('~/Downloads'))

cust_file = [f for f in os.listdir() if 'customer' in f.lower()][0]
trans_file = [f for f in os.listdir() if 'trans' in f.lower()][0]

customers = pd.read_csv(cust_file) if cust_file.endswith('.csv') else pd.read_excel(cust_file)
transactions = pd.read_csv(trans_file) if trans_file.endswith('.csv') else pd.read_excel(trans_file)

transactions['TRANSACTION_VALUE'] = pd.to_datetime(transactions['date_new'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
transactions['Sum_m'] = transactions['Sum_payment']

conn = sqlite3.connect(':memory:')
customers.to_sql('customers', conn, index=False, if_exists='replace')
transactions.to_sql('transactions', conn, index=False, if_exists='replace')

print("✅ DONE.")

import os
import pandas as pd
import sqlite3

os.chdir(os.path.expanduser('~/Downloads'))

cust_file = [f for f in os.listdir() if 'customer' in f.lower()][0]
trans_file = [f for f in os.listdir() if 'trans' in f.lower()][0]

customers = pd.read_csv(cust_file) if cust_file.endswith('.csv') else pd.read_excel(cust_file)
transactions = pd.read_csv(trans_file) if trans_file.endswith('.csv') else pd.read_excel(trans_file)

transactions['TRANSACTION_VALUE'] = pd.to_datetime(transactions['date_new'], dayfirst=True, format='mixed').dt.strftime('%Y-%m-%d %H:%M:%S')
transactions['Sum_m'] = transactions['Sum_payment']

conn = sqlite3.connect(':memory:')
customers.to_sql('customers', conn, index=False, if_exists='replace')
transactions.to_sql('transactions', conn, index=False, if_exists='replace')


check = pd.read_sql("SELECT MIN(TRANSACTION_VALUE) as start_date, MAX(TRANSACTION_VALUE) as end_date, COUNT(DISTINCT strftime('%Y-%m', TRANSACTION_VALUE)) as total_months FROM transactions", conn)
print(check)

✅ DONE.
            start_date             end_date  total_months
0  2015-06-01 00:00:00  2016-06-01 00:00:00            13


In [10]:
# === Task 1: Continuous Activity Analysis (12/12 Months) ===
# Extraction of customers with a continuous 12-month transaction history, total operations, overall average check, and average monthly spend.

q1 = """
WITH monthly_act AS (
    SELECT 
        ID_client,
        COUNT(DISTINCT strftime('%Y-%m', TRANSACTION_VALUE)) AS active_months,
        COUNT(*) AS total_ops,
        AVG(Sum_m) AS avg_check,
        SUM(Sum_m) / 12.0 AS avg_monthly_spend
    FROM transactions
    WHERE TRANSACTION_VALUE >= '2015-06-01' AND TRANSACTION_VALUE < '2016-06-01'
    GROUP BY ID_client
    HAVING active_months = 12
)
SELECT 
    c.Id_client,
    ROUND(m.avg_check, 2) AS avg_check_period,
    ROUND(m.avg_monthly_spend, 2) AS avg_monthly_spend,
    m.total_ops AS total_ops_period
FROM monthly_act m
JOIN customers c ON m.ID_client = c.Id_client;
"""

df1 = pd.read_sql(q1, conn)
df1


,Id_client,avg_check_period,avg_monthly_spend,total_ops_period
0,16052,8.96,45262.80,60605
1,185122,7.15,173.49,291
2,185151,8.91,272.56,367
3,185156,11.75,776.79,793
4,185348,7.47,628.51,1009
...,...,...,...,...
77,283183,26.49,1063.87,482
78,283367,7.00,334.47,573
79,283480,8.55,733.97,1030
80,396473,10.78,951.88,1060


In [11]:
# === Task 2: Monthly Operational Metrics & Annual Shares ===
# Monthly aggregation of transaction metrics (average check, operations per client, active clients) and their percentage shares relative to annual totals.

q2 = """
WITH monthly AS (
    SELECT 
        strftime('%Y-%m', TRANSACTION_VALUE) AS month_yr,
        COUNT(*) AS month_ops,
        SUM(Sum_m) AS month_sum,
        AVG(Sum_m) AS avg_check_month,
        COUNT(DISTINCT ID_client) AS active_clients
    FROM transactions
    WHERE TRANSACTION_VALUE >= '2015-06-01' AND TRANSACTION_VALUE < '2016-06-01'
    GROUP BY strftime('%Y-%m', TRANSACTION_VALUE)
),
totals AS (
    SELECT 
        COUNT(*) AS year_ops,
        SUM(Sum_m) AS year_sum
    FROM transactions
    WHERE TRANSACTION_VALUE >= '2015-06-01' AND TRANSACTION_VALUE < '2016-06-01'
)
SELECT 
    m.month_yr AS "Месяц",
    ROUND(m.avg_check_month, 2) AS "Средняя сумма чека",
    ROUND(CAST(m.month_ops AS FLOAT) / m.active_clients, 2) AS "Среднее кол-во операций в месяц",
    m.active_clients AS "Клиентов совершавших операции",
    ROUND(m.month_ops * 100.0 / t.year_ops, 2) AS "Доля операций от года (%)",
    ROUND(m.month_sum * 100.0 / t.year_sum, 2) AS "Доля суммы от года (%)"
FROM monthly m
CROSS JOIN totals t
ORDER BY m.month_yr;
"""

df2 = pd.read_sql(q2, conn)
df2

,Месяц,Средняя сумма чека,Среднее кол-во операций в месяц,Клиентов совершавших операции,Доля операций от года (%),Доля суммы от года (%)
0,2015-06,9.35,14.37,224,0.84,0.83
1,2015-07,9.26,31.63,939,7.79,7.61
2,2015-08,9.06,31.85,907,7.58,7.24
3,2015-09,9.28,31.17,901,7.37,7.21
4,2015-10,9.32,30.69,967,7.78,7.65
5,2015-11,9.23,29.78,918,7.17,6.98
6,2015-12,9.32,29.89,1032,8.09,7.96
7,2016-01,9.06,30.66,991,7.97,7.62
8,2016-02,10.01,38.42,1254,12.63,13.35
9,2016-03,9.90,36.64,1181,11.35,11.85


In [12]:
# === Task 3: Gender Demographic Breakdown & Spending Shares ===
# Monthly distribution of active clients and overall expenditure categorized by gender (Male, Female, Unspecified/NA).

q3 = """
SELECT 
    strftime('%Y-%m', t.TRANSACTION_VALUE) AS month_yr,
    
    -- Доля клиентов (%)
    ROUND(COUNT(DISTINCT CASE WHEN UPPER(c.Gender) = 'M' THEN t.ID_client END) * 100.0 / COUNT(DISTINCT t.ID_client), 2) AS pct_clients_M,
    ROUND(COUNT(DISTINCT CASE WHEN UPPER(c.Gender) = 'F' THEN t.ID_client END) * 100.0 / COUNT(DISTINCT t.ID_client), 2) AS pct_clients_F,
    ROUND(COUNT(DISTINCT CASE WHEN c.Gender IS NULL OR UPPER(c.Gender) NOT IN ('M', 'F') THEN t.ID_client END) * 100.0 / COUNT(DISTINCT t.ID_client), 2) AS pct_clients_NA,
    
    -- Доля затрат (%)
    ROUND(SUM(CASE WHEN UPPER(c.Gender) = 'M' THEN t.Sum_m ELSE 0 END) * 100.0 / SUM(t.Sum_m), 2) AS spend_share_M,
    ROUND(SUM(CASE WHEN UPPER(c.Gender) = 'F' THEN t.Sum_m ELSE 0 END) * 100.0 / SUM(t.Sum_m), 2) AS spend_share_F,
    ROUND(SUM(CASE WHEN c.Gender IS NULL OR UPPER(c.Gender) NOT IN ('M', 'F') THEN t.Sum_m ELSE 0 END) * 100.0 / SUM(t.Sum_m), 2) AS spend_share_NA
FROM transactions t
LEFT JOIN customers c ON t.ID_client = c.Id_client
WHERE t.TRANSACTION_VALUE >= '2015-06-01' AND t.TRANSACTION_VALUE < '2016-06-01'
GROUP BY strftime('%Y-%m', t.TRANSACTION_VALUE)
ORDER BY month_yr;
"""

df3 = pd.read_sql(q3, conn)
df3

,month_yr,pct_clients_M,pct_clients_F,pct_clients_NA,spend_share_M,spend_share_F,spend_share_NA
0,2015-06,31.25,65.63,3.13,25.31,72.52,2.17
1,2015-07,31.42,65.60,2.98,24.60,73.27,2.13
2,2015-08,30.43,66.26,3.31,21.40,76.24,2.36
3,2015-09,30.19,65.82,4.00,22.40,74.89,2.71
4,2015-10,30.51,66.80,2.69,24.59,73.27,2.14
5,2015-11,29.19,67.76,3.05,24.20,73.44,2.36
6,2015-12,31.40,65.60,3.00,27.07,70.47,2.45
7,2016-01,30.58,66.70,2.72,26.58,70.78,2.65
8,2016-02,30.06,66.83,3.11,26.44,70.80,2.76
9,2016-03,30.74,66.55,2.71,24.83,72.16,3.01


In [8]:
# === Task 4: Age Group Stratification & Quarterly Trend Analysis ===
# Customer segmentation into 10-year age groups (including NA) with key metrics evaluated both overall and broken down by quarters.

q4 = """
WITH age_prep AS (
    SELECT 
        c.Id_client,
        CASE 
            WHEN c.Age IS NULL OR c.Age = '' THEN 'NA'
            ELSE CAST((CAST(c.Age AS INT) / 10) * 10 AS TEXT) || '-' || CAST((CAST(c.Age AS INT) / 10) * 10 + 9 AS TEXT)
        END AS age_group,
        t.Sum_m,
        strftime('%Y', t.TRANSACTION_VALUE) || '-Q' || ((CAST(strftime('%m', t.TRANSACTION_VALUE) AS INT) + 2) / 3) AS quarter
    FROM transactions t
    LEFT JOIN customers c ON t.ID_client = c.Id_client
    WHERE t.TRANSACTION_VALUE >= '2015-06-01' AND t.TRANSACTION_VALUE < '2016-06-01'
)
SELECT 
    age_group,
    'ALL_PERIOD' AS period,
    ROUND(SUM(Sum_m), 2) AS total_sum,
    COUNT(*) AS total_ops,
    ROUND(AVG(Sum_m), 2) AS avg_check,
    ROUND(SUM(Sum_m) * 100.0 / (SELECT SUM(Sum_m) FROM age_prep), 2) AS sum_share_pct
FROM age_prep
GROUP BY age_group

UNION ALL

SELECT 
    age_group,
    quarter AS period,
    ROUND(SUM(Sum_m), 2) AS total_sum,
    COUNT(*) AS total_ops,
    ROUND(AVG(Sum_m), 2) AS avg_check,
    ROUND(SUM(Sum_m) * 100.0 / (SELECT SUM(Sum_m) FROM age_prep), 2) AS sum_share_pct
FROM age_prep
GROUP BY age_group, quarter
ORDER BY age_group, period;
"""

df4 = pd.read_sql(q4, conn)
df4

,age_group,period,total_sum,total_ops,avg_check,sum_share_pct
0,0-9,2016-Q1,6549.82,593,11.05,0.28
1,0-9,ALL_PERIOD,6549.82,593,11.05,0.28
2,10-19,2016-Q1,78533.20,8132,9.66,3.36
3,10-19,ALL_PERIOD,78533.20,8132,9.66,3.36
4,20-29,2016-Q1,495088.05,49357,10.03,21.21
5,20-29,ALL_PERIOD,495088.05,49357,10.03,21.21
6,30-39,2016-Q1,457502.88,47963,9.54,19.60
7,30-39,ALL_PERIOD,457502.88,47963,9.54,19.60
8,40-49,2016-Q1,300687.99,29538,10.18,12.88
9,40-49,ALL_PERIOD,300687.99,29538,10.18,12.88
